# Query Classifier (Router) Training

Trains AlephBERT-base for 3-way query classification (`wiki`, `kz`, `knesset`).

**Input**: `hsrc_train.jsonl` with query + case_name  
**Output**: Checkpoint in `./hebrew_cls_ckpts_case_only/`  
**Result**: ~97.6% macro-F1

In [28]:
# train_eval_hebrew_cls_clean_case_only.py
# Fine-tune a Hebrew 3-way query classifier (knesset/kz/wiki) using ONLY case_name labels.
#
# Deps: pip install transformers scikit-learn torch joblib
# GPU recommended (works on CPU too).

import os, json, time, random
from pathlib import Path
from typing import Dict, Iterable, List, Tuple, Optional

import numpy as np
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report, confusion_matrix
)
from sklearn.model_selection import StratifiedShuffleSplit

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
)

# --------------------------- Config ---------------------------
TRAIN_JSONL   = os.getenv("TRAIN_JSONL", "./hsrc_train.jsonl")
SAVE_DIR      = os.getenv("SAVE_DIR",   "./hebrew_cls_ckpts_case_only")

# Models to try (order matters for logs). Add others if you like.
MODEL_CANDIDATES = [
    "onlplab/alephbert-base",            # strong Hebrew encoder
    # "microsoft/mdeberta-v3-base",      # multilingual alternative (optional)
]

# Train/loop knobs
BATCH_TRAIN   = int(os.getenv("BATCH_TRAIN",  "32"))
BATCH_EVAL    = int(os.getenv("BATCH_EVAL",   "64"))
EPOCHS        = int(os.getenv("EPOCHS",       "5"))
EARLY_STOP    = int(os.getenv("EARLY_STOP",   "2"))      # patience on macro-F1
GRAD_CLIP     = float(os.getenv("GRAD_CLIP",  "1.0"))
USE_AMP       = bool(int(os.getenv("USE_AMP", "0")))     # default off (often safer here)
SEED          = int(os.getenv("SEED",         "42"))
MAX_LEN       = int(os.getenv("MAX_LEN",      "128"))
LR            = float(os.getenv("LR",         "2e-5"))
WARMUP_FRAC   = float(os.getenv("WARMUP_FRAC","0.05"))
LABEL_SMOOTH  = float(os.getenv("LABEL_SMOOTH","0.05"))
ALPHA_CW      = float(os.getenv("ALPHA_CW",   "0.15"))   # 0=none, 1=full inv-freq

NUM_WORKERS   = int(os.getenv("NUM_WORKERS",  "1"))

# >>> 0.3 / 0.7 stratified validation/train split <<<
VAL_FRAC      = float(os.getenv("VAL_FRAC", "0.3"))  # 30% validation, 70% train

# --------------------------------------------------------------

LABEL_ALIASES = {
    "mafat_retrieval_knesset_corpus": "knesset", "knesset": "knesset",
    "mafat_retrieval_kz_corpus": "kz", "kz": "kz", "kol-zchut": "kz",
    "mafat_retrieval_wikipedia_corpus": "wiki", "wikipedia": "wiki", "wiki": "wiki",
}

def seed_all(seed:int=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = False
seed_all()

def norm_case(x: Optional[str]) -> Optional[str]:
    if not x: return None
    return LABEL_ALIASES.get(str(x).strip(), None)

def iter_train_rows(path:str) -> Iterable[dict]:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            j = json.loads(line)
            yield {
                "query": j.get("query",""),
                "case_name": norm_case(j.get("case_name")),
            }

# --------------------------- Data ---------------------------
def load_xy_case_only(path:str) -> Tuple[List[str], List[str]]:
    X, y = [], []
    for r in iter_train_rows(path):
        if r["case_name"] and r["query"].strip():
            X.append(r["query"]); y.append(r["case_name"])
    return X, y

# --------------------------- Torch bits ---------------------------
class QDataset(Dataset):
    def __init__(self, texts:List[str], y_ids:List[int], tok, max_len:int):
        self.texts, self.y, self.tok, self.max_len = texts, y_ids, tok, max_len
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        enc = self.tok(self.texts[i], truncation=True, padding="max_length",
                       max_length=self.max_len, return_tensors="pt")
        out = {k:v.squeeze(0) for k,v in enc.items()}
        out["labels"] = torch.tensor(self.y[i], dtype=torch.long)
        return out

def blended_weights(y_ids:List[int], num_labels:int, alpha:float) -> torch.Tensor:
    if alpha <= 0: return torch.ones(num_labels, dtype=torch.float32)
    cnt = np.bincount(np.array(y_ids), minlength=num_labels).astype(np.float32)
    inv = cnt.sum() / np.maximum(cnt, 1.0)
    inv = inv / inv.mean()
    w = 1.0 + alpha * (inv - 1.0)
    return torch.tensor(w, dtype=torch.float32)

@torch.no_grad()
def evaluate(model, loader, device, id2label):
    model.eval()
    y_true, y_pred = [], []
    for batch in loader:
        batch = {k:v.to(device) for k,v in batch.items()}
        logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
        pred = logits.argmax(dim=-1)
        y_true += batch["labels"].cpu().tolist()
        y_pred += pred.cpu().tolist()
    y_true_lab = [id2label[i] for i in y_true]
    y_pred_lab = [id2label[i] for i in y_pred]
    macro = f1_score(y_true_lab, y_pred_lab, average="macro")
    acc   = accuracy_score(y_true_lab, y_pred_lab)
    return macro, acc, y_true_lab, y_pred_lab

def train_one(model_name:str, X_tr, y_tr, X_va, y_va, labels:list) -> Dict:
    label2id = {l:i for i,l in enumerate(labels)}
    id2label = {i:l for l,i in label2id.items()}

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    y_tr_ids = [label2id[y] for y in y_tr]; y_va_ids = [label2id[y] for y in y_va]

    ds_tr = QDataset(X_tr, y_tr_ids, tok, MAX_LEN)
    ds_va = QDataset(X_va, y_va_ids, tok, MAX_LEN)
    dl_tr = DataLoader(ds_tr, batch_size=BATCH_TRAIN, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
    dl_va = DataLoader(ds_va, batch_size=BATCH_EVAL,  shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(labels), id2label=id2label, label2id=label2id
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    total_steps = EPOCHS * max(1, len(dl_tr))
    sch = get_linear_schedule_with_warmup(opt, int(total_steps*WARMUP_FRAC), total_steps)

    cw = blended_weights(y_tr_ids, len(labels), alpha=ALPHA_CW).to(device)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and device.type=="cuda")

    best_macro, best_state = -1.0, None
    patience = EARLY_STOP

    for ep in range(1, EPOCHS+1):
        model.train(); t0=time.time(); running=0.0
        for batch in dl_tr:
            batch = {k:v.to(device) for k,v in batch.items()}
            with torch.amp.autocast("cuda", enabled=USE_AMP and device.type=="cuda"):
                logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
                K = logits.size(-1); y = batch["labels"]
                with torch.no_grad():
                    sm = torch.zeros_like(logits)
                    sm.fill_(LABEL_SMOOTH / (K-1))
                    sm.scatter_(1, y.view(-1,1), 1.0 - LABEL_SMOOTH)
                logp = torch.log_softmax(logits, dim=-1)
                loss = (-sm * logp * cw.view(1,-1)).sum(dim=-1).mean()

            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True); sch.step()
            running += float(loss.item())

        macro, acc, y_true_lab, y_pred_lab = evaluate(model, dl_va, device, id2label)
        dt = time.time()-t0
        print(f"[{model_name}] ep {ep}/{EPOCHS}  loss={running/max(1,len(dl_tr)):.4f}  "
              f"val_macro={macro:.4f}  val_acc={acc:.4f}  time={dt:.1f}s")

        if macro > best_macro:
            best_macro = macro; best_state = {k:v.detach().cpu() for k,v in model.state_dict().items()}
            patience = EARLY_STOP
        else:
            patience -= 1
            if patience <= 0:
                print("  early stop.")
                break

    # Save best
    out_dir = Path(SAVE_DIR) / f"{Path(model_name).name}_case_name_best"
    out_dir.mkdir(parents=True, exist_ok=True)
    if best_state: model.load_state_dict(best_state)
    model.save_pretrained(out_dir); tok.save_pretrained(out_dir)
    with open(out_dir/"labels.json","w",encoding="utf-8") as f:
        json.dump({"labels": labels}, f, ensure_ascii=False, indent=2)

    # Final detailed report
    dl_va = DataLoader(ds_va, batch_size=BATCH_EVAL, shuffle=False, num_workers=NUM_WORKERS)
    macro, acc, y_true_lab, y_pred_lab = evaluate(model, dl_va, device, id2label)
    print("\n" + classification_report(y_true_lab, y_pred_lab, digits=3, labels=labels))
    print("Confusion (rows=true, cols=pred):", labels)
    print(confusion_matrix(y_true_lab, y_pred_lab, labels=labels))
    print(f"[Saved best] {out_dir} (macro-F1={best_macro:.4f})")

    return {
        "model": model_name, "dir": str(out_dir),
        "macro": float(macro), "acc": float(acc),
        "labels": labels
    }

def stratified_val_train_split(X: List[str], y: List[str], val_frac: float, seed: int):
    """
    Create a stratified split with val_frac for validation and (1 - val_frac) for train.
    Returns X_tr, X_va, y_tr, y_va where len(X_va) ~= val_frac * len(X).
    """
    sss = StratifiedShuffleSplit(n_splits=1, test_size=val_frac, random_state=seed)
    (train_idx, val_idx), = sss.split(X, y)
    X_tr = [X[i] for i in train_idx]
    y_tr = [y[i] for i in train_idx]
    X_va = [X[i] for i in val_idx]
    y_va = [y[i] for i in val_idx]
    # quick report
    def dist(labels):
        uniq, cnt = np.unique(labels, return_counts=True)
        return {u: int(c) for u,c in zip(uniq, cnt)}
    print(f"Stratified split: val_frac={val_frac:.2f}  ->  train={len(X_tr)}  val={len(X_va)}")
    print("  train label dist:", dist(y_tr))
    print("  val   label dist:", dist(y_va))
    return X_tr, X_va, y_tr, y_va

def main():
    assert Path(TRAIN_JSONL).exists(), f"Missing {TRAIN_JSONL}"
    Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

    X, y = load_xy_case_only(TRAIN_JSONL)
    labels = sorted(set(y))
    print(f"Loaded {len(X):,} queries. Label dist: " + str({c:y.count(c) for c in labels}))

    # 0.7 / 0.3 stratified validation/train split
    X_tr, X_va, y_tr, y_va = stratified_val_train_split(X, y, val_frac=VAL_FRAC, seed=SEED)

    results = []
    for m in MODEL_CANDIDATES:
        try:
            results.append(train_one(m, X_tr, y_tr, X_va, y_va, labels))
        except Exception as e:
            print(f"[SKIP] {m}: {e}")

    if not results:
        print("No runs completed."); return
    results.sort(key=lambda d: d["macro"], reverse=True)
    print("\n=== Leaderboard (macro-F1) ===")
    for r in results:
        print(f"{Path(r['dir']).name:40s}  macro={r['macro']:.4f}  acc={r['acc']:.4f}  ({r['model']})")

if __name__ == "__main__":
    main()


Loaded 2,034 queries. Label dist: {'knesset': 476, 'kz': 804, 'wiki': 754}
Stratified split: val_frac=0.30  ->  train=1423  val=611
  train label dist: {np.str_('knesset'): 333, np.str_('kz'): 562, np.str_('wiki'): 528}
  val   label dist: {np.str_('knesset'): 143, np.str_('kz'): 242, np.str_('wiki'): 226}


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at onlplab/alephbert-base and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[onlplab/alephbert-base] ep 1/5  loss=0.5706  val_macro=0.9525  val_acc=0.9558  time=3.1s
[onlplab/alephbert-base] ep 2/5  loss=0.2722  val_macro=0.9680  val_acc=0.9722  time=3.3s
[onlplab/alephbert-base] ep 3/5  loss=0.2450  val_macro=0.9760  val_acc=0.9787  time=3.1s
[onlplab/alephbert-base] ep 4/5  loss=0.2372  val_macro=0.9726  val_acc=0.9755  time=3.1s
[onlplab/alephbert-base] ep 5/5  loss=0.2355  val_macro=0.9694  val_acc=0.9722  time=3.2s
  early stop.

              precision    recall  f1-score   support

     knesset      0.939     0.972     0.955       143
          kz      0.983     0.975     0.979       242
        wiki      1.000     0.987     0.993       226

    accuracy                          0.979       611
   macro avg      0.974     0.978     0.976       611
weighted avg      0.979     0.979     0.979       611

Confusion (rows=true, cols=pred): ['knesset', 'kz', 'wiki']
[[139   4   0]
 [  6 236   0]
 [  3   0 223]]
[Saved best] hebrew_cls_ckpts_case_only/alephber